Import Libraries

In [2]:
import pandas as pd
import numpy as np

Extract USER-DRUG Interations

In [3]:
prec=pd.read_csv(r"...mimic iv\mimic-iv-3.1\hosp\prescriptions.csv.gz")
prec.head()

C:\Users\local-vdi-admin\AppData\Local\Temp\3\ipykernel_117808\3024448914.py:1: DtypeWarning: Columns (11) have mixed types. Specify dtype option on import or set low_memory=False.
  prec=pd.read_csv(r"C:\Users\local-vdi-admin\Desktop\mimic iv\mimic-iv-3.1\hosp\prescriptions.csv.gz")


,subject_id,hadm_id,pharmacy_id,poe_id,poe_seq,order_provider_id,starttime,stoptime,drug_type,drug,...,gsn,ndc,prod_strength,form_rx,dose_val_rx,dose_unit_rx,form_val_disp,form_unit_disp,doses_per_24_hrs,route
0,10000032,22595853,12775705,10000032-55,55.0,P85UQ1,2180-05-08 08:00:00,2180-05-07 22:00:00,MAIN,Furosemide,...,008209,5.107901e+10,40mg Tablet,NaN,40,mg,1,TAB,1.0,PO/NG
1,10000032,22595853,18415984,10000032-42,42.0,P23SJA,2180-05-07 02:00:00,2180-05-07 22:00:00,MAIN,Ipratropium Bromide Neb,...,021700,4.879801e+08,2.5mL Vial,NaN,1,NEB,1,VIAL,4.0,IH
2,10000032,22595853,23637373,10000032-35,35.0,P23SJA,2180-05-07 01:00:00,2180-05-07 09:00:00,MAIN,Furosemide,...,008208,5.107901e+10,20mg Tablet,NaN,20,mg,1,TAB,1.0,PO/NG
3,10000032,22595853,26862314,10000032-41,41.0,P23SJA,2180-05-07 01:00:00,2180-05-07 01:00:00,MAIN,Potassium Chloride,...,001275,2.450041e+08,10mEq ER Tablet,NaN,40,mEq,4,TAB,1.0,PO
4,10000032,22595853,30740602,10000032-27,27.0,P23SJA,2180-05-07 00:00:00,2180-05-07 22:00:00,MAIN,Sodium Chloride 0.9% Flush,...,NaN,0.000000e+00,10 mL Syringe,NaN,3,mL,0.3,SYR,3.0,IV


In [4]:
diagnoses=pd.read_csv(r"....mimic iv\mimic-iv-3.1\hosp\diagnoses_icd.csv.gz")

diagnoses.head()

,subject_id,hadm_id,seq_num,icd_code,icd_version
0,10000032,22595853,1,5723,9
1,10000032,22595853,2,78959,9
2,10000032,22595853,3,5715,9
3,10000032,22595853,4,07070,9
4,10000032,22595853,5,496,9


In [5]:
diagnoses_d=pd.read_csv(r"....mimic iv\mimic-iv-3.1\hosp\d_icd_diagnoses.csv.gz")

diagnoses_d.head()

,icd_code,icd_version,long_title
0,0010,9,Cholera due to vibrio cholerae
1,0011,9,Cholera due to vibrio cholerae el tor
2,0019,9,"Cholera, unspecified"
3,0020,9,Typhoid fever
4,0021,9,Paratyphoid fever A


In [8]:
diag = pd.read_csv(
    r"....mimic iv\mimic-iv-3.1\hosp\diagnoses_icd.csv.gz",
    usecols=["subject_id", "hadm_id", "icd_code", "icd_version", "seq_num"]
)

diag["icd_code"] = diag["icd_code"].astype(str).str.upper().str.strip()




TARGET_ICD9 = (
    "140","141","142","143","144","145","146","147","148","149",
    "153","154","162","174","185","188"
)
TARGET_ICD10 = (
    "C00","C01","C02","C03","C04","C05","C06","C07","C08",
    "C18","C19","C20","C34","C50","C61","C67"
)





cancer_mask = (
    ((diag.icd_version == 9) & diag.icd_code.str.startswith(TARGET_ICD9)) |
    ((diag.icd_version == 10) & diag.icd_code.str.startswith(TARGET_ICD10))
)

cancer_diag = diag[cancer_mask].copy()

patients_with_cancer = cancer_diag["hadm_id"].unique()

print(f"[INFO] Cancer admissions: {len(patients_with_cancer)}")


[INFO] Cancer admissions: 21769


In [7]:
# =========================================================
# Build time-ordered drug sequences per HADM_ID (LSTM-ready)
# (UPDATED: drug2idx uses drug NAMES, PAD-safe, DDI-ready)
# =========================================================

import pandas as pd
import numpy as np
import torch
from collections import defaultdict
from torch.nn.utils.rnn import pad_sequence

# -----------------------------
# INPUT DATA (assumed loaded)
# -----------------------------
# prec : prescriptions.csv
# patients_with_cancer : np.array of HADM_IDs



# -----------------------------
# SETTINGS
# -----------------------------
MAX_GAP_HOURS = 24      # collapse repeated drugs within 24h
MIN_SEQ_LEN = 2         # drop admissions with <2 drugs
PAD_TOKEN = "<PAD>"

# -----------------------------
# Filter prescriptions
# -----------------------------
prec_f = prec[prec["hadm_id"].isin(patients_with_cancer)].copy()

prec_f = prec_f[[
    "subject_id",
    "hadm_id",
    "drug",
    "starttime"
]].dropna()

prec_f["starttime"] = pd.to_datetime(prec_f["starttime"])

# Normalize drug names
prec_f["drug"] = (
    prec_f["drug"]
    .str.lower()
    .str.strip()
)

# -----------------------------
# Sort temporally
# -----------------------------
prec_f = prec_f.sort_values(["hadm_id", "starttime"])

# -----------------------------
# Build drug vocabulary (NAME-BASED, PAD SAFE)
# -----------------------------
drug_vocab = sorted(prec_f["drug"].unique().tolist())
drug_vocab = [PAD_TOKEN] + drug_vocab

drug2idx = {drug: i for i, drug in enumerate(drug_vocab)}
idx2drug = {i: drug for drug, i in drug2idx.items()}

NUM_DRUGS = len(drug2idx)
print(f"[INFO] Unique drugs (incl PAD): {NUM_DRUGS}")

# Encode drugs using name-based mapping
prec_f["drug_id"] = prec_f["drug"].map(drug2idx)

# -----------------------------
# Build time-ordered sequences
# -----------------------------
drug_seqs = defaultdict(list)
time_seqs = defaultdict(list)

for _, row in prec_f.iterrows():
    hid = row["hadm_id"]
    drug_seqs[hid].append(row["drug_id"])
    time_seqs[hid].append(row["starttime"])

# -----------------------------
# Collapse duplicates in short windows
# -----------------------------
final_seqs = {}
final_hadm_ids = []

for hid, drugs in drug_seqs.items():
    times = time_seqs[hid]

    seq = [drugs[0]]
    last_time = times[0]

    for d, t in zip(drugs[1:], times[1:]):
        if d != seq[-1] or (t - last_time).total_seconds() > MAX_GAP_HOURS * 3600:
            seq.append(d)
            last_time = t

    if len(seq) >= MIN_SEQ_LEN:
        final_seqs[hid] = seq
        final_hadm_ids.append(hid)

print(f"[INFO] Admissions with valid drug sequences: {len(final_seqs)}")

# -----------------------------
# Convert to PyTorch tensors
# -----------------------------
X_drug = [
    torch.tensor(seq, dtype=torch.long)
    for seq in final_seqs.values()
]

hadm_ids = np.array(final_hadm_ids)

# -----------------------------
# Collate function for LSTM
# -----------------------------
def collate_drug_sequences(batch):
    lengths = torch.tensor([len(seq) for seq in batch])
    padded = pad_sequence(batch, batch_first=True, padding_value=0)  # 0 = <PAD>
    return padded, lengths

# -----------------------------
# Example batch (sanity check)
# -----------------------------
padded, lengths = collate_drug_sequences(X_drug[:4])

print("[INFO] Example padded batch shape:", padded.shape)
print("[INFO] Example sequence lengths:", lengths.tolist())

# Decode example to verify correctness
print("[INFO] Example decoded sequence:")
print([idx2drug[d] for d in X_drug[0].tolist()])

# -----------------------------
# OUTPUT OBJECTS
# -----------------------------
# X_drug        → list[Tensor(seq_len)]  (drug indices, PAD-safe)
# hadm_ids      → np.array
# drug2idx      → maps drug_name → index (USED FOR DDI)
# idx2drug      → reverse map
# NUM_DRUGS     → vocab size

print("\n[READY] Drug sequences are LSTM / Transformer + DDI ready ✅")


[INFO] Unique drugs (incl PAD): 3318


KeyboardInterrupt: 

In [8]:
# =========================================================
# SAVE DRUG SEQUENCES (LSTM-ready, ORDER-SAFE, DDI-aligned)
# =========================================================

import numpy as np
import torch
import pickle
from torch.nn.utils.rnn import pad_sequence

# ------------------------------
# OUTPUT PATHS
# ------------------------------
BASE_PATH = r"....new_cancer_drug_sequences.npy"
DRUG2IDX_PATH = r"....new_drug2idx.pkl"

# ------------------------------
# Ensure deterministic ordering
# ------------------------------
hadm_ids = np.array(sorted(final_seqs.keys()))

drug_seqs = [
    torch.tensor(final_seqs[hid], dtype=torch.long)
    for hid in hadm_ids
]

# ------------------------------
# Map hadm_id → subject_id
# ------------------------------
hadm_to_subject = (
    prec_f[["hadm_id", "subject_id"]]
    .drop_duplicates()
    .set_index("hadm_id")["subject_id"]
    .to_dict()
)

subject_ids = np.array([hadm_to_subject[hid] for hid in hadm_ids])

# ------------------------------
# Pad sequences
# ------------------------------
lengths = np.array([len(seq) for seq in drug_seqs])

X = pad_sequence(
    drug_seqs,
    batch_first=True,
    padding_value=0        # 0 = <PAD>
).numpy()

# ------------------------------
# SAVE OUTPUTS
# ------------------------------
np.save(BASE_PATH, X)
np.save(BASE_PATH.replace(".npy", "_lengths.npy"), lengths)
np.save(BASE_PATH.replace(".npy", "_hadm_ids.npy"), hadm_ids)
np.save(BASE_PATH.replace(".npy", "_subject_ids.npy"), subject_ids)

# Save drug vocabulary (CRITICAL for DDI + decoding)
with open(DRUG2IDX_PATH, "wb") as f:
    pickle.dump(drug2idx, f)

print(f"\n✅ Saved {len(hadm_ids)} admission drug sequences")
print("Drug tensor shape:", X.shape)
print("Lengths shape:", lengths.shape)
print("Saved sequences to:", BASE_PATH)
print("Saved drug2idx to:", DRUG2IDX_PATH)



✅ Saved 21267 admission drug sequences
Drug tensor shape: (21267, 1282)
Lengths shape: (21267,)
Saved sequences to: C:\Temp\hemat_cancer_drug_sequences.npy
Saved drug2idx to: C:\Temp\hemat_drug2idx.pkl
